#### ***Hybrid Search***

##### ***Hybrid search is a technique combine two or more search algorithms to improve accuracy and relevance of search results.***

#### ***Hybrid Search = Vector Search + BM25 Search***

#### ***1.BM25 Search***
##### ***It is used to search exact keyword matchings like Abbrevations and Commnads***

In [154]:
### Load the environment variables

from dotenv import load_dotenv

load_dotenv()

True

In [155]:
###path Exists
import os
path = "../kubernetes"

if os.path.exists(path):
    print("Path Is Exist.")
else:
    print("Path is not Exist.")

Path Is Exist.


In [156]:
### Load all pdf files using DirectoryLoader by PyMuPdfLoader

from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()


print("Number Of Documents:",len(documents))

Number Of Documents: 3983


In [157]:
### Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 9507


In [158]:
### BM25 Retriever

from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks
)

In [159]:
bm25_retriever.k=5

In [160]:
query = "What is a Kubernetes Deployment?"
retrieved_docs = bm25_retriever.invoke(query)
retrieved_docs


[Document(metadata={'producer': 'WeasyPrint 56.1', 'creator': '', 'creationdate': '', 'source': '..\\kubernetes\\Concepts.pdf', 'file_path': '..\\kubernetes\\Concepts.pdf', 'total_pages': 676, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 6}, page_content="spec:\n      containers:\n      - name: nginx\n        image: nginx:1.14.2\n        ports:\n        - containerPort: 80\nOne way to create a Deployment using a manifest file like the one above is to use the kubectl \napply command in the kubectl command-line interface, passing the .yaml file as an argument.\nHere's an example:\nkubectl apply -f https://k8s.io/examples/application/deployment.yaml\nThe output is similar to this:\ndeployment.apps/nginx-deployment created\nRequired fields\nIn the manifest (YAML or JSON file) for the Kubernetes object you want to create, you'll need to\nset values for the following fields:\napiVersion

In [161]:
for i,doc in enumerate(retrieved_docs,start=1):
    print(f"------ Document {i}--------")

    print("Page Content: ",doc.page_content)
    print("Source:",doc.metadata.get("source"))
    print("Page:",doc.metadata.get("page"))

    print("#"*60)

------ Document 1--------
Page Content:  spec:
      containers:
      - name: nginx
        image: nginx:1.14.2
        ports:
        - containerPort: 80
One way to create a Deployment using a manifest file like the one above is to use the kubectl 
apply command in the kubectl command-line interface, passing the .yaml file as an argument.
Here's an example:
kubectl apply -f https://k8s.io/examples/application/deployment.yaml
The output is similar to this:
deployment.apps/nginx-deployment created
Required fields
In the manifest (YAML or JSON file) for the Kubernetes object you want to create, you'll need to
set values for the following fields:
apiVersion - Which version of the Kubernetes API you're using to create this object
kind - What kind of object you want to create
metadata - Data that helps uniquely identify the object, including a name string, UID, and
optional namespace
spec - What state you desire for the object
The precise format of the object spec is different for every Ku

In [162]:
test_queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?"
]

for query in test_queries:
    retrieved_docs = bm25_retriever.invoke(query)
    print("Query:",query)
    for i,doc in enumerate(retrieved_docs,start=1):
        print(f"------ Document {i}--------")

        print("Page Content: ",doc.page_content)
        print("Source:",doc.metadata.get("source"))
        print("Page:",doc.metadata.get("page"))

        print("#"*60)
    print("="*50)



Query: What is a Kubernetes Deployment?
------ Document 1--------
Page Content:  spec:
      containers:
      - name: nginx
        image: nginx:1.14.2
        ports:
        - containerPort: 80
One way to create a Deployment using a manifest file like the one above is to use the kubectl 
apply command in the kubectl command-line interface, passing the .yaml file as an argument.
Here's an example:
kubectl apply -f https://k8s.io/examples/application/deployment.yaml
The output is similar to this:
deployment.apps/nginx-deployment created
Required fields
In the manifest (YAML or JSON file) for the Kubernetes object you want to create, you'll need to
set values for the following fields:
apiVersion - Which version of the Kubernetes API you're using to create this object
kind - What kind of object you want to create
metadata - Data that helps uniquely identify the object, including a name string, UID, and
optional namespace
spec - What state you desire for the object
The precise format of t

Query: What is a Kubernetes Service?
------ Document 1--------
Page Content:  Debug Running Pods
Get a Shell to a Running Container
Debug Pods
This guide is to help users debug applications that are deployed into Kubernetes and not
behaving correctly. This is not a guide for people who want to debug their cluster. For that you
should check out this guide.
Diagnosing the problem
The first step in troubleshooting is triage. What is the problem? Is it your Pods, your
Replication Controller or your Service?
Debugging Pods
Debugging Replication Controllers
Debugging Services
Debugging Pods
The first step in debugging a Pod is taking a look at it. Check the current state of the Pod and
recent events with the following command:
kubectl describe pods ${POD_NAME}
Look at the state of the containers in the pod. Are they all Running? Have there been recent
restarts?
Continue debugging depending on the state of the pods.
My pod stays pending
If a Pod is stuck in Pending it means that it can not be

In [163]:
evaluation_data = [
    {
        "query": "What is a Kubernetes Deployment?",
        "relevant_pages": [5, 9]
    },
    {
        "query": "What is a Kubernetes Pod?",
        "relevant_pages": [85]
    },
    {
        "query": "What is a Kubernetes Service?",
        "relevant_pages": [228, 17]
    },
    {
        "query": "What is a ReplicaSet?",
        "relevant_pages": [156, 164]
    },
    {
        "query": "What is a ConfigMap?",
        "relevant_pages": [392, 360, 375]
    }
]

In [164]:
def recall_at_k(retrieved_pages,relevant_pages):

    retrieved_pages = set(retrieved_pages)
    relevant_pages = set(relevant_pages)

    if not relevant_pages:
        return 0

    
    return len(relevant_pages & retrieved_pages)/len(relevant_pages)

In [165]:
def precision_at_k(retrieved_pages,relevant_pages):

    relevant_pages = set(relevant_pages)

    if not relevant_pages:
        return 0
    counter = 0
    for page in retrieved_pages:

        if page in relevant_pages:
            counter+=1

    return counter/len(retrieved_pages)


In [166]:
def mrr_at_k(retrieved_pages,relevant_pages):

    rr = 0
    for rank,page in enumerate(retrieved_pages,start=1):

        if page in relevant_pages:
            rr = 1/rank
            break

    return rr

In [167]:
for evaluate_data in evaluation_data:
    query = evaluate_data['query']
    relevant_pages = evaluate_data['relevant_pages']
    print("Query:",query)
    print("Relevant Pages:",relevant_pages)

    ## BM25 Retriever

    results = bm25_retriever.invoke(query)
    retrieved_pages = []
    for doc in results:
        page = doc.metadata.get("page")
        retrieved_pages.append(int(page))
    print("Retrieved Pages:",retrieved_pages)
    recall_score = recall_at_k(retrieved_pages,relevant_pages)
    precision_score = precision_at_k(retrieved_pages,relevant_pages)
    mrr_score = mrr_at_k(retrieved_pages,relevant_pages)
    print("Recall Score:",recall_score)
    print("Precision Score:",precision_score)
    print("MRR Score:",mrr_score)

    

Query: What is a Kubernetes Deployment?
Relevant Pages: [5, 9]
Retrieved Pages: [6, 85, 3, 449, 59]
Recall Score: 0.0
Precision Score: 0.0
MRR Score: 0
Query: What is a Kubernetes Pod?
Relevant Pages: [85]
Retrieved Pages: [85, 85, 6, 3, 449]
Recall Score: 1.0
Precision Score: 0.4
MRR Score: 1.0
Query: What is a Kubernetes Service?
Relevant Pages: [228, 17]
Retrieved Pages: [408, 413, 6, 85, 3]
Recall Score: 0.0
Precision Score: 0.0
MRR Score: 0
Query: What is a ReplicaSet?
Relevant Pages: [156, 164]
Retrieved Pages: [6, 85, 217, 408, 449]
Recall Score: 0.0
Precision Score: 0.0
MRR Score: 0
Query: What is a ConfigMap?
Relevant Pages: [392, 360, 375]
Retrieved Pages: [6, 85, 217, 408, 449]
Recall Score: 0.0
Precision Score: 0.0
MRR Score: 0


#### **vector Search**

In [168]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [169]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)

In [170]:
similarity_retriever = vectorstore.as_retriever(search_type = "similarity",
        search_kwargs = {"k":5}
)

#### **Build Hybrid Search**

In [171]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[similarity_retriever,bm25_retriever],
    weights=[0.7,0.3]
)

In [172]:
query = "What is a Kubernetes Deployment?"

hybrid_docs = hybrid_retriever.invoke(query,k=5)

for i,doc in enumerate(hybrid_docs,start=1):
    print(f"---------- Document {i}------------")
    print("Page Content:",doc.page_content)
    print("Source:",doc.metadata.get("source"))
    print("Page:",doc.metadata.get("page"))

    print("="*60)

---------- Document 1------------
Page Content: Kubernetes is a portable, extensible, open source platform for managing containerized
workloads and services, that facilitates both declarative configuration and automation. It has a
large, rapidly growing ecosystem. Kubernetes services, support, and tools are widely available.
The name Kubernetes originates from Greek, meaning helmsman or pilot. K8s as an
abbreviation results from counting the eight letters between the "K" and the "s". Google open-
sourced the Kubernetes project in 2014. Kubernetes combines over 15 years of Google's
experience running production workloads at scale with best-of-breed ideas and practices from
the community.
Going back in time
Let's take a look at why Kubernetes is so useful by going back in time.
Deployment evolution
Traditional deployment era: Early on, organizations ran applications on physical servers.
There was no way to define resource boundaries for applications in a physical server, and this
Source:

In [173]:
## Metric Evaluation
for evaluate_data in evaluation_data:
    query = evaluate_data['query']
    relevant_pages = evaluate_data['relevant_pages']
    print("Query:",query)
    print("Relevant Pages:",relevant_pages)

    ## Hybrid Retriever

    results = hybrid_retriever.invoke(query)[:5]
    retrieved_pages = []
    for doc in results:
        page = doc.metadata.get("page")
        retrieved_pages.append(int(page))
    print("Retrieved Pages:",retrieved_pages)
    recall_score = recall_at_k(retrieved_pages,relevant_pages)
    precision_score = precision_at_k(retrieved_pages,relevant_pages)
    mrr_score = mrr_at_k(retrieved_pages,relevant_pages)
    print("Recall Score:",recall_score)
    print("Precision Score:",precision_score)
    print("MRR Score:",mrr_score)

    

Query: What is a Kubernetes Deployment?
Relevant Pages: [5, 9]


Retrieved Pages: [1, 9, 5, 8, 6]
Recall Score: 1.0
Precision Score: 0.4
MRR Score: 0.5
Query: What is a Kubernetes Pod?
Relevant Pages: [85]
Retrieved Pages: [85, 1, 12, 12, 13]
Recall Score: 1.0
Precision Score: 0.2
MRR Score: 1.0
Query: What is a Kubernetes Service?
Relevant Pages: [228, 17]
Retrieved Pages: [228, 1, 17, 16, 147]
Recall Score: 1.0
Precision Score: 0.4
MRR Score: 1.0
Query: What is a ReplicaSet?
Relevant Pages: [156, 164]
Retrieved Pages: [164, 156, 164, 214, 362]
Recall Score: 1.0
Precision Score: 0.6
MRR Score: 1.0
Query: What is a ConfigMap?
Relevant Pages: [392, 360, 375]
Retrieved Pages: [392, 360, 375, 618, 935]
Recall Score: 1.0
Precision Score: 0.6
MRR Score: 1.0


In [174]:
#Modify the Weights to 0.8,0.2

from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever_1 = EnsembleRetriever(
    retrievers=[similarity_retriever,bm25_retriever],
    weights=[0.8,0.2]
)

In [175]:
## Metric Evaluation
for evaluate_data in evaluation_data:
    query = evaluate_data['query']
    relevant_pages = evaluate_data['relevant_pages']
    print("Query:",query)
    print("Relevant Pages:",relevant_pages)

    ## Hybrid Retriever

    results = hybrid_retriever_1.invoke(query)[:5]
    retrieved_pages = []
    for doc in results:
        page = doc.metadata.get("page")
        retrieved_pages.append(int(page))
    print("Retrieved Pages:",retrieved_pages)
    recall_score = recall_at_k(retrieved_pages,relevant_pages)
    precision_score = precision_at_k(retrieved_pages,relevant_pages)
    mrr_score = mrr_at_k(retrieved_pages,relevant_pages)
    print("Recall Score:",recall_score)
    print("Precision Score:",precision_score)
    print("MRR Score:",mrr_score)

    

Query: What is a Kubernetes Deployment?
Relevant Pages: [5, 9]
Retrieved Pages: [1, 9, 5, 8, 6]
Recall Score: 1.0
Precision Score: 0.4
MRR Score: 0.5
Query: What is a Kubernetes Pod?
Relevant Pages: [85]
Retrieved Pages: [85, 1, 12, 12, 13]
Recall Score: 1.0
Precision Score: 0.2
MRR Score: 1.0
Query: What is a Kubernetes Service?
Relevant Pages: [228, 17]
Retrieved Pages: [228, 1, 17, 16, 147]
Recall Score: 1.0
Precision Score: 0.4
MRR Score: 1.0
Query: What is a ReplicaSet?
Relevant Pages: [156, 164]
Retrieved Pages: [164, 156, 164, 214, 362]
Recall Score: 1.0
Precision Score: 0.6
MRR Score: 1.0
Query: What is a ConfigMap?
Relevant Pages: [392, 360, 375]
Retrieved Pages: [392, 360, 375, 618, 935]
Recall Score: 1.0
Precision Score: 0.6
MRR Score: 1.0


In [176]:
##Design a Prompt
from langchain_core.prompts import ChatPromptTemplate
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template("""
You are a Kubernetes documentation Assistant.

Answer the question using only the provided Context only.


Rules:
    1.Don't use information outside the Context.
    2.If the answer is not available in the context,
    say I don't have information based on the Provided Documents.


Context:
{context}

Question:
{question}

Answer:
""")

In [177]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025C9FA15310>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025C9FAB04A0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [178]:
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
rag_chain = ({
    "context":hybrid_retriever | RunnableLambda(format_docs),
    "question":RunnablePassthrough()
}
|prompt
| llm 
| StrOutputParser()
)

In [179]:
query = "What is a Kubernetes Deployment?"

In [180]:
hybrid_documents = rag_chain.invoke(query)
hybrid_documents

'A **Kubernetes Deployment** is an API object that represents an application you want to run on a cluster.  \nWhen you create a Deployment you provide a **spec** that describes the desired state—e.g., the container image to use and how many replica Pods should be running. The Kubernetes control plane reads this spec, creates the requested Pods, and continuously monitors the **status**. If the actual state diverges from the desired state (for example, a Pod fails), the system automatically corrects the difference by starting replacement Pods or updating existing ones. Deployments are managed through the `kubectl` command‑line tool.'

In [181]:
test_queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?"
]

In [182]:
results = []
for query in test_queries:
    response = rag_chain.invoke(query)
    results.append({
        "Query":query,
        "Response":response
    })

In [183]:
for result in results:

    print("Question:",result['Query'])
    print("Response:",result['Response'])
    print("#"*70)

Question: What is a Kubernetes Deployment?
Response: A **Kubernetes Deployment** is a Kubernetes API object that describes the desired state for an application running in the cluster. By defining a Deployment spec (including details such as the number of replicas, container images, and other settings), you tell Kubernetes how many instances of the application should be running and how they should be configured. The Kubernetes control plane continuously compares the actual state of those Pods to the spec; if a Pod fails or the spec changes, the Deployment controller creates, updates, or replaces Pods to reconcile the difference and keep the cluster in the desired state.
######################################################################
Question: What is a Kubernetes Pod?
Response: A **Kubernetes Pod** is the smallest deployable unit in Kubernetes. It is a logical host that groups **one or more containers** (such as Docker containers) together, providing them with **shared storage (v

In [184]:

##citations
def get_context_with_source(retrieved_docs):
    context_parts = []

    for i,doc in enumerate(retrieved_docs,start=1):

        content = doc.page_content

        source = doc.metadata.get("source")

        page = doc.metadata.get("page")

        context_parts.append(
            f""" 
        Document {i}
        Content:{content}
        Source: {source}
        Page:{page} """
        )

    context = "\n\n".join(context_parts)

    return context

In [185]:
test_queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?"
]

In [186]:
##Design a Prompt
from langchain_core.prompts import ChatPromptTemplate
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template("""
You are a Kubernetes documentation Assistant.

Answer the question using only the provided Context only.


Rules:
    1.Don't use information outside the Context.
    2.If the answer is not available in the context,
    say I don't have information based on the Provided Documents.
    3.After the answer provide the source used.
    4.just Include the source file name and page number without extra information.


Context:
{context}

Question:
{question}

Answer:
""")

In [187]:
for query in test_queries:

    retrieved_docs = hybrid_retriever.invoke(query)
    context = get_context_with_source(retrieved_docs)
    message = prompt.invoke({"context":context,"question":query})

    response = llm.invoke(message)

    print("Query:",query)
    print("Response:",response.content)
    print("#"*60)

Query: What is a Kubernetes Deployment?
Response: A Kubernetes Deployment is a Kubernetes API object that represents an application running in the cluster. It defines the desired state of that application (such as the number of pod replicas) in its spec. The Kubernetes control plane continuously compares the spec with the actual status and automatically creates, updates, or replaces pods to match the desired state—handling scaling, rolling updates, and recovery from failures.

**Sources**  
..\\kubernetes\\Tutorials.pdf page 9  
..\\kubernetes\\Concepts.pdf page 5
############################################################
Query: What is a Kubernetes Pod?
Response: A Kubernetes Pod is the smallest deployable unit in Kubernetes—a group of one or more application containers (such as Docker) that share storage, an IP address, and other network resources, together with a specification that defines how the containers should run. The containers in a Pod are always co‑located, co‑scheduled, 